In [45]:
import onnxruntime
import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the class labels in order
class_labels = ['metal', 'plastic', 'glass', 'biodegradable']

def preprocess_image(image_path):
    """
    Preprocess the image exactly as MobileNetV3 expects
    """
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image)
    return image_tensor.unsqueeze(0).numpy()

def read_label_file(label_path):
    """
    Read the first value from label file with better error handling
    """
    try:
        with open(label_path, 'r') as f:
            content = f.read().strip().split()
            if not content:
                raise ValueError("Empty label file")
            value = float(content[0])
            if not (0 <= value < len(class_labels)):
                raise ValueError(f"Label value {value} out of range")
            return int(value)
    except Exception as e:
        raise ValueError(f"Error reading label file: {str(e)}")

def plot_confusion_matrix(confusion_matrix, class_names):
    """
    Create and save a visual confusion matrix
    """
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix, 
                annot=True, 
                fmt='d', 
                cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)
    
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    
    # Save the plot
    plt.savefig('confusion_matrix.png')
    plt.close()

def main():
    current_dir = Path.cwd()
    images_dir = current_dir / 'GARBAGE_YOLO'/'test' / 'images'
    labels_dir = current_dir / 'GARBAGE_YOLO'/'test' / 'labels'
    model_path = current_dir / 'waste_classifier.onnx'
    
    print("Initializing model...")
    session = onnxruntime.InferenceSession(model_path)
    input_name = session.get_inputs()[0].name
    
    results = []
    errors = []
    total_images = len(list(images_dir.glob('*.jpg')))
    processed = 0
    
    print(f"Processing {total_images} images...")
    
    # Initialize confusion matrix
    confusion_mat = np.zeros((len(class_labels), len(class_labels)), dtype=int)
    
    for image_path in images_dir.glob('*.jpg'):
        try:
            # Get corresponding label file
            label_path = labels_dir / f"{image_path.stem}.txt"
            
            if not label_path.exists():
                errors.append(f"Missing label file for {image_path.name}")
                continue
                
            # Read true label first
            true_class = read_label_file(label_path)
            
            # Read image and predict
            input_data = preprocess_image(image_path)
            output = session.run(None, {input_name: input_data})
            probabilities = torch.nn.functional.softmax(torch.tensor(output[0]), dim=1)
            predicted_class = torch.argmax(probabilities, dim=1).item()
            
            # Update confusion matrix
            confusion_mat[true_class][predicted_class] += 1
            
            # Store result
            results.append({
                'image': image_path.name,
                'true_class': class_labels[true_class],
                'predicted_class': class_labels[predicted_class],
                'correct': predicted_class == true_class
            })
            
            processed += 1
            if processed % 100 == 0:
                print(f"Processed {processed}/{total_images} images")
                
        except Exception as e:
            errors.append(f"Error processing {image_path.name}: {str(e)}")
            continue
    
    # Create DataFrame and calculate metrics
    df = pd.DataFrame(results)
    accuracy = (df['correct'].sum() / len(df)) * 100
    
    print("\n=== Classification Results ===")
    print(f"Total images processed: {len(df)}")
    print(f"Failed images: {len(errors)}")
    print(f"Correct predictions: {df['correct'].sum()}")
    print(f"Accuracy: {accuracy:.2f}%")
    
    # Class-wise metrics
    print("\nClass-wise Performance:")
    for class_name in class_labels:
        class_data = df[df['true_class'] == class_name]
        if not class_data.empty:
            class_accuracy = (class_data['correct'].sum() / len(class_data)) * 100
            print(f"{class_name}: {class_accuracy:.2f}% ({len(class_data)} samples)")
    
    # Display confusion matrix as text
    print("\nConfusion Matrix:")
    confusion_df = pd.DataFrame(
        confusion_mat,
        index=class_labels,
        columns=class_labels
    )
    print(confusion_df)
    
    # Plot visual confusion matrix
    plot_confusion_matrix(confusion_mat, class_labels)
    print("\nConfusion matrix visualization saved as 'confusion_matrix.png'")
    
    # Save results
    results_file = current_dir / 'prediction_results.csv'
    df.to_csv(results_file, index=False)
    print(f"Detailed results saved to: {results_file}")
    
    # Save error log if there are errors
    if errors:
        error_file = current_dir / 'error_log.txt'
        with open(error_file, 'w') as f:
            for error in errors:
                f.write(f"{error}\n")
        print(f"Error log saved to: {error_file}")

if __name__ == "__main__":
    main()

Initializing model...
Processing 1042 images...
Processed 100/1042 images

=== Classification Results ===
Total images processed: 182
Failed images: 860
Correct predictions: 5
Accuracy: 2.75%

Class-wise Performance:
metal: 0.00% (3 samples)
plastic: 0.00% (9 samples)
biodegradable: 2.94% (170 samples)

Confusion Matrix:
               metal  plastic  glass  biodegradable
metal              0        1      1              1
plastic            0        0      0              9
glass              0        0      0              0
biodegradable    150       11      4              5

Confusion matrix visualization saved as 'confusion_matrix.png'
Detailed results saved to: c:\Users\divya\OneDrive\Desktop\ECS\prediction_results.csv
Error log saved to: c:\Users\divya\OneDrive\Desktop\ECS\error_log.txt
